In [2]:
!pip install yfinance pandas

import yfinance as yf
import pandas as pd


[notice] A new release of pip is available: 23.3.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [11]:
def analyze_stock(ticker, period="1y"):

    df = yf.Ticker(ticker).history(period=period)

    if df.empty:
        print(f"{ticker}: No data found.\n")
        return

    # ------------------------------------------------------
    # ICHIMOKU LINES (trend framework)
    # ------------------------------------------------------
    df['conversion'] = (df['High'].rolling(9).max() + df['Low'].rolling(9).min()) / 2     # Tenkan
    df['base'] = (df['High'].rolling(26).max() + df['Low'].rolling(26).min()) / 2         # Kijun
    df['spanA'] = ((df['conversion'] + df['base']) / 2).shift(26)                        # Leading Span A
    df['spanB'] = ((df['High'].rolling(52).max() + df['Low'].rolling(52).min()) / 2).shift(26) # Leading Span B
    df['chikou'] = df['Close'].shift(-26)                                                # Lagging span

    c = df['Close'].iloc[-1]       # current price
    conv = df['conversion'].iloc[-1]
    base = df['base'].iloc[-1]
    spanA = df['spanA'].iloc[-1]
    spanB = df['spanB'].iloc[-1]

    cloud_color = "BULLISH (Green)" if spanA > spanB else "BEARISH (Red)"

    # ------------------------------------------------------
    # PRICE LOCATION VS CLOUD
    # ------------------------------------------------------
    if c > spanA and c > spanB:
        price_zone = "Above Cloud (Strong Bullish)"
        market_mode = "BULL"
    elif c < spanA and c < spanB:
        price_zone = "Below Cloud (Strong Bearish)"
        market_mode = "BEAR"
    else:
        price_zone = "Inside Cloud (No Direction)"
        market_mode = "NEUTRAL"

    # ------------------------------------------------------
    # TK CROSS (momentum check)
    # ------------------------------------------------------
    if conv > base:
        tk = "Bullish TK Cross — rising momentum"
    elif conv < base:
        tk = "Bearish TK Cross — weakening momentum"
    else:
        tk = "Neutral TK Cross"

    # ------------------------------------------------------
    # ICHIMOKU SUMMARY SIGNAL
    # ------------------------------------------------------
    if (market_mode == "BULL" and conv > base and spanA > spanB):
        ichi_signal = "STRONG BUY"
    elif (market_mode == "BEAR" and conv < base and spanA < spanB):
        ichi_signal = "STRONG SELL"
    else:
        ichi_signal = "NEUTRAL / WAIT"

    # ------------------------------------------------------
    # ATR-BASED BUY/SELL LEVELS (improved logic)
    # ------------------------------------------------------
    # ATR is used to avoid false entries due to volatility spikes
    df['ATR'] = (df['High'] - df['Low']).rolling(14).mean()
    atr = df['ATR'].iloc[-1]

    # 🔥 Better buy logic → “pullback to Kijun” entry
    if market_mode == "BULL":
        aggressive_buy = c  # buy at current price
        conservative_buy = base  # kijun pullback (fair value zone)
        suggestion = (
            f"Aggressive Buy (trend): {aggressive_buy:.2f}\n"
            f"Conservative Buy (pullback): {conservative_buy:.2f}"
        )

    elif market_mode == "BEAR":
        aggressive_sell = c
        conservative_sell = base  # kijun pullback
        suggestion = (
            f"Aggressive Sell (trend): {aggressive_sell:.2f}\n"
            f"Conservative Sell (pullback): {conservative_sell:.2f}"
        )

    else:
        suggestion = "Price inside cloud — no safe buy/sell level."

    # ------------------------------------------------------
    # RSI (overbought/oversold filter)
    # ------------------------------------------------------
    delta = df['Close'].diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)

    avg_gain = gain.rolling(14).mean()
    avg_loss = loss.rolling(14).mean()

    rs = avg_gain.iloc[-1] / avg_loss.iloc[-1] if avg_loss.iloc[-1] != 0 else 999
    rsi = 100 - (100 / (1 + rs))

    if rsi > 70:
        rsi_status = "Overbought — avoid buying"
    elif rsi < 30:
        rsi_status = "Oversold — dip buying zone"
    else:
        rsi_status = "Normal"

    # ------------------------------------------------------
    # MACD (momentum confirmation)
    # ------------------------------------------------------
    ema12 = df['Close'].ewm(span=12, adjust=False).mean()
    ema26 = df['Close'].ewm(span=26, adjust=False).mean()
    macd = ema12 - ema26
    signal = macd.ewm(span=9, adjust=False).mean()
    macd_hist = macd - signal

    macd_desc = (
        "Positive → bullish momentum"
        if macd_hist.iloc[-1] > 0 else
        "Negative → bearish momentum"
    )

    # ------------------------------------------------------
    # PRINT RESULTS
    # ------------------------------------------------------
    print("\n-----------------------------------------------")
    print(f"📈 {ticker} — Analysis")
    print("-----------------------------------------------")
    print(f"Price: {c:.2f}")
    print(f"Cloud: {cloud_color}")
    print(f"Position: {price_zone}")
    print(f"TK Cross: {tk}")
    print(f"Ichimoku Signal: {ichi_signal}")
    print(f"Action: {suggestion}")
    print(f"RSI: {rsi:.2f} → {rsi_status}")
    print(f"MACD Hist: {macd_hist.iloc[-1]:.4f} → {macd_desc}")
    print("-----------------------------------------------\n")


tickers = ["SNPS", "AAPL", "TSLA", "MSFT"]  # Add as many as you want

for t in tickers:
    analyze_stock(t)




-----------------------------------------------
📈 SNPS — Analysis
-----------------------------------------------
Price: 389.83
Cloud: BEARISH (Red)
Position: Below Cloud (Strong Bearish)
TK Cross: Bearish TK Cross — weakening momentum
Ichimoku Signal: STRONG SELL
Action: Aggressive Sell (trend): 389.83
Conservative Sell (pullback): 436.90
RSI: 18.10 → Oversold — dip buying zone
MACD Hist: -2.4827 → Negative → bearish momentum
-----------------------------------------------


-----------------------------------------------
📈 AAPL — Analysis
-----------------------------------------------
Price: 272.41
Cloud: BULLISH (Green)
Position: Above Cloud (Strong Bullish)
TK Cross: Bullish TK Cross — rising momentum
Ichimoku Signal: STRONG BUY
Action: Aggressive Buy (trend): 272.41
Conservative Buy (pullback): 260.41
RSI: 60.98 → Normal
MACD Hist: -0.2647 → Negative → bearish momentum
-----------------------------------------------


-----------------------------------------------
📈 TSLA — Anal